# Flood Prediction Collaboration Notebook

This Colab-ready notebook is designed as a **systematic collaboration document** for reviewing the current flood prediction baseline, the dataset pipeline, the modeling outputs, and the remaining blockers.

It combines:
- dataset context
- extraction workflow notes from the Caravan paper
- model metrics
- multiple evaluation visualizations
- practical next steps for the team


## 1. Collaboration Objective

The goal of this notebook is to give collaborators one place where they can:

1. understand the current data and model pipeline,
2. review the model's behavior through multiple visualizations,
3. understand how extraction and standardization work in large-sample hydrology datasets,
4. see what is implemented versus what is still blocked,
5. decide on the next model upgrade path.


## 2. Research Paper Notes: How the Extraction Workflow Works

Reference paper used for interpretation:
- **Kratzert et al. (2023)**, *Caravan - a global community dataset for large-sample hydrology*, Scientific Data.

### Key takeaways relevant to this project

The paper explains a **standardized extraction workflow** for large-sample hydrology datasets:

- **Meteorological forcing source**: Caravan uses **ERA5-Land** because it has global coverage, sub-daily resolution, and cloud availability.
- **Time handling matters**: streamflow observations are usually recorded in **local gauge time**, while global meteorological data are often stored in **UTC/GMT+0**.
- **Extraction step**: hourly meteorological data are first spatially aggregated over the catchment, then shifted into the gauge's local time zone before daily aggregates are computed.
- **Catchment attributes**: static basin attributes are derived from **HydroATLAS** and additional climate indices are derived from the extracted time series.
- **Cloud processing**: the paper highlights **Google Earth Engine** as a practical way to process large geospatial datasets without forcing every user to download and manage huge raster archives locally.
- **Extensibility requirement**: to add a new basin, the critical ingredients are a **catchment boundary shapefile** and a **streamflow time series in local time**.
- **Validation logic**: the authors statistically compare extracted forcing products against existing trusted datasets to ensure that the standardized extraction process is not introducing unreasonable distortion.

### Why this matters for our flood prediction work

This paper is useful because it shows that a flood or hydrology model should not just collect data casually. It should:

- align forcing data with gauge-local timing,
- aggregate gridded data over actual basin geometry,
- preserve metadata and provenance,
- validate extracted variables against trusted references,
- and make the pipeline reproducible.

That is also the direction for upgrading the current baseline model from a simple discharge-memory model into a stronger rainfall-driven flood prediction workflow.


## 3. Current Project Status

### Fully implemented
- CAMELS-US static basin attributes downloaded and integrated
- USGS Water Data API daily discharge downloaded and integrated
- Local SQLite database built
- Baseline next-day discharge prediction model trained
- Evaluation dashboard and plots created

### Partially implemented
- NASA GPM IMERG Early V07 metadata access is integrated
- Official NASA file links are verified
- Direct binary file ingestion is blocked in the current environment by `401 Unauthorized`

### Interpretation
The current system is a **working predictive baseline**, but it is **not yet a full rainfall-driven flood prediction system** because real precipitation forcing has not been ingested into the model features yet.


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11


In [ ]:
def resolve_repo_root():
    candidates = [
        Path.cwd(),
        Path.cwd() / 'Flood-Prediction-Model---CloudAngles',
        Path('/content/Flood-Prediction-Model---CloudAngles'),
        Path('/content/drive/MyDrive/Flood-Prediction-Model---CloudAngles')
    ]
    for candidate in candidates:
        if (candidate / 'flood_prediction_modeling' / 'outputs' / 'predictions.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root with prediction outputs.')

repo_root = resolve_repo_root()
model_dir = repo_root / 'flood_prediction_modeling'
outputs_dir = model_dir / 'outputs'
raw_dir = model_dir / 'data' / 'raw'

pred_path = outputs_dir / 'predictions.csv'
metrics_path = outputs_dir / 'metrics.json'
nasa_status_path = raw_dir / 'nasa' / 'imerg_download_attempt.json'
usgs_path = raw_dir / 'usgs' / 'daily_discharge.csv'

print('Repository root:', repo_root)
print('Predictions file:', pred_path)


In [ ]:
pred_df = pd.read_csv(pred_path)
pred_df['date'] = pd.to_datetime(pred_df['date'])

with open(metrics_path, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

with open(nasa_status_path, 'r', encoding='utf-8') as f:
    nasa_status = json.load(f)

usgs_df = pd.read_csv(usgs_path)
usgs_df['date'] = pd.to_datetime(usgs_df['date'])

pred_df['residual'] = pred_df['predicted'] - pred_df['actual']
pred_df['abs_error'] = pred_df['residual'].abs()
pred_df['rolling_mae_14'] = pred_df['abs_error'].rolling(14, min_periods=1).mean()

pred_df.head()


## 4. Model Metrics Summary

These are the saved metrics from the current baseline model run.


In [ ]:
summary_df = pd.DataFrame([
    ['USGS Site', metrics['site_id']],
    ['Train Rows', metrics['train_rows']],
    ['Test Rows', metrics['test_rows']],
    ['RMSE', round(metrics['rmse'], 2)],
    ['MAE', round(metrics['mae'], 2)],
    ['NSE', round(metrics['nse'], 4)],
    ['USGS Daily Rows', len(usgs_df)],
    ['NASA IMERG File Access', f"HTTP {nasa_status.get('status')}"]
], columns=['Metric', 'Value'])

summary_df


## 5. Evaluation Visualizations

The following plots are included to make model review collaborative and interpretable. Each chart highlights a different performance behavior.


In [ ]:
# 1. Actual vs predicted over time
fig, ax = plt.subplots(figsize=(14, 5))
window = pred_df.tail(180)
ax.plot(window['date'], window['actual'], label='Actual discharge', color='#1f77b4', linewidth=2)
ax.plot(window['date'], window['predicted'], label='Predicted discharge', color='#2ca02c', linewidth=2)
ax.set_title('Actual vs Predicted Daily Discharge (Last 180 Test Days)')
ax.set_xlabel('Date')
ax.set_ylabel('Discharge')
ax.legend()
plt.show()


In [ ]:
# 2. Residuals over time
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.bar(window['date'], window['residual'], color=np.where(window['residual'] >= 0, '#ffb000', '#d62728'))
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Residuals Over Time (Predicted - Actual)')
ax.set_xlabel('Date')
ax.set_ylabel('Residual')
plt.show()


In [ ]:
# 3. Actual vs predicted scatter
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(pred_df['actual'], pred_df['predicted'], alpha=0.45, color='#17becf', edgecolors='none')
lims = [min(pred_df['actual'].min(), pred_df['predicted'].min()), max(pred_df['actual'].max(), pred_df['predicted'].max())]
ax.plot(lims, lims, linestyle='--', color='black', label='Ideal fit')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_title('Actual vs Predicted Scatter')
ax.set_xlabel('Actual discharge')
ax.set_ylabel('Predicted discharge')
ax.legend()
plt.show()


In [ ]:
# 4. Error distribution histogram
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.hist(pred_df['residual'], bins=35, color='#9467bd', alpha=0.85)
ax.axvline(0, color='black', linestyle='--')
ax.set_title('Distribution of Prediction Residuals')
ax.set_xlabel('Residual (Predicted - Actual)')
ax.set_ylabel('Frequency')
plt.show()


In [ ]:
# 5. Rolling 14-day MAE
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(pred_df['date'], pred_df['rolling_mae_14'], color='#ff7f0e', linewidth=2)
ax.set_title('Rolling 14-Day Mean Absolute Error')
ax.set_xlabel('Date')
ax.set_ylabel('Rolling 14-day MAE')
plt.show()


In [ ]:
# 6. Monthly average actual vs predicted comparison
monthly = pred_df.set_index('date')[['actual', 'predicted']].resample('M').mean().reset_index()
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(monthly['date'], monthly['actual'], label='Monthly avg actual', color='#1f77b4', linewidth=2)
ax.plot(monthly['date'], monthly['predicted'], label='Monthly avg predicted', color='#2ca02c', linewidth=2)
ax.set_title('Monthly Average Actual vs Predicted Discharge')
ax.set_xlabel('Month')
ax.set_ylabel('Average discharge')
ax.legend()
plt.show()


## 6. Interpretation for Team Discussion

### What these plots show

- **Time-series forecast plot**: shows whether the model tracks broad hydrological behavior and large events.
- **Residual plot**: shows when the model over-predicts or under-predicts.
- **Scatter plot**: shows overall fit strength and how tightly predictions cluster around the ideal line.
- **Histogram**: shows whether errors are centered or biased.
- **Rolling 14-day MAE**: shows whether accuracy deteriorates during unstable periods such as sharp flow changes.
- **Monthly average comparison**: shows whether the model captures slower seasonal behavior.

### Practical reading of the current baseline

The baseline model performs strongly for a first-pass discharge forecast, especially in terms of NSE. However, the plots also make it clear that the model becomes less stable during abrupt changes in discharge, which is exactly where real precipitation forcing would likely help.


## 7. What Has Been Done So Far

Use this section directly in discussions if needed.

### Completed work
- Identified credible flood and hydrology datasets.
- Built a local dataset procurement package.
- Designed a database model for hydrology and flood prediction data.
- Downloaded and integrated CAMELS-US basin attributes.
- Downloaded and integrated USGS daily discharge data.
- Verified NASA IMERG metadata access and official granule links.
- Built a local SQLite database.
- Trained a baseline next-day discharge prediction model.
- Generated metrics and multiple evaluation visualizations.
- Created a local team dashboard page and pushed the project to GitHub.

### Current model answer
Yes, there is a real model. It is not just a web page or a dataset dump. The current implementation is a **baseline predictive model** that forecasts next-day discharge using historical discharge behavior and basin attributes.


## 8. Recommended Next Steps

1. Add authenticated NASA IMERG precipitation ingestion.
2. Rebuild the feature set with rainfall forcing aligned to basin-local time.
3. Expand from one gauge to multiple basins for broader generalization.
4. Compare the baseline linear model against stronger machine learning models.
5. Add event-oriented flood metrics if the target evolves from discharge forecasting to flood classification or flood-threshold exceedance prediction.


## 9. How to Use This in Google Colab

### Option A: Open directly from GitHub
- Upload this notebook to the repository
- Open it in Colab using the GitHub notebook picker

### Option B: Upload notebook and repo files to Colab
- Upload the notebook
- Upload or mount the repository folder
- Run all cells

### Expected repository structure
This notebook expects the repository to contain:
- `flood_prediction_modeling/outputs/predictions.csv`
- `flood_prediction_modeling/outputs/metrics.json`
- `flood_prediction_modeling/data/raw/nasa/imerg_download_attempt.json`
- `flood_prediction_modeling/data/raw/usgs/daily_discharge.csv`
